# Conversation Summary Memory

Neste notebook vamos estudar **Summary Memory**, utilizando:

- Python
- LangChain
- Ollama
- Qwen 3 4B
- `ConversationSummaryMemory`

A ideia principal é entender como o histórico de uma conversa pode ser **resumido automaticamente**, em vez de manter todas as mensagens completas.

> **Importante:** `ConversationSummaryMemory` é uma API clássica/legada do LangChain. Como a aula utiliza essa abordagem, vamos reproduzi-la com `langchain_classic`.


## 1. O problema do histórico completo

Na `ConversationBufferMemory`, podemos armazenar todas as mensagens:

```text
Usuário → mensagem 1
IA     → resposta 1
Usuário → mensagem 2
IA     → resposta 2
Usuário → mensagem 3
IA     → resposta 3
...
```

Em conversas longas, esse histórico pode ficar grande.

A `ConversationSummaryMemory` utiliza uma estratégia diferente:

```text
Histórico
   ↓
LLM resume o histórico
   ↓
Resumo
   ↓
Novo contexto para a conversa
```

Em vez de enviar todas as mensagens completas, a aplicação mantém um resumo do que aconteceu.


## 2. Buffer, Window e Summary

Até agora estudamos três estratégias.

### Buffer Memory

Mantém o histórico inteiro:

```text
M1 M2 M3 M4 M5 M6 M7 M8
```

### Window Memory

Mantém somente uma janela recente:

```text
M1 M2 M3 M4 M5 M6 M7 M8
               └───────┘
                 janela
```

### Summary Memory

Condensa o histórico:

```text
M1 M2 M3 M4 M5 M6 M7 M8
          ↓
     "Resumo da conversa"
```

A Summary Memory tenta preservar o **significado e os principais fatos** da conversa, sem necessariamente manter cada mensagem literalmente.


## 3. Importando as classes

Como estamos usando a API clássica do LangChain, `ConversationSummaryMemory` e `ConversationChain` são importados de `langchain_classic`.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationChain


## 4. Configurando o Ollama

A memória de resumo precisa de um LLM para gerar e atualizar o resumo.

Por isso, passamos o nosso `ChatOllama` para a memória.

Neste projeto, o próprio Qwen 3 4B será utilizado para resumir a conversa.


In [ ]:
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)


## 5. Criando a ConversationSummaryMemory

Aqui está o ponto principal:

```python
memory = ConversationSummaryMemory(
    llm=llm,
    return_messages=True
)
```

O parâmetro:

```python
llm=llm
```

informa qual modelo será responsável por criar e atualizar os resumos.

Isso é importante porque a Summary Memory precisa fazer chamadas ao modelo para transformar o histórico em um resumo.


In [ ]:
memory = ConversationSummaryMemory(
    llm=llm,
    return_messages=True
)


## 6. Criando a ConversationChain

Agora conectamos o modelo à memória.


In [ ]:
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)


## 7. Primeira interação

Vamos informar algumas informações sobre o usuário.

Depois dessa interação, a memória pode começar a construir seu contexto resumido.


In [ ]:
resposta = conversation.invoke(
    "Meu nome é Felipe e estou estudando inteligência artificial."
)

print(resposta["response"])


## 8. Segunda interação

Agora adicionamos mais informações relacionadas aos estudos.


In [ ]:
resposta = conversation.invoke(
    "Estou estudando Python, LangChain e desenvolvimento de aplicações com LLMs."
)

print(resposta["response"])


## 9. Terceira interação

Vamos adicionar informações sobre as ferramentas utilizadas.


In [ ]:
resposta = conversation.invoke(
    "Para meus estudos, estou usando Ollama com o modelo Qwen 3 4B localmente."
)

print(resposta["response"])


## 10. Quarta interação

Adicionamos mais uma informação sobre o objetivo dos estudos.


In [ ]:
resposta = conversation.invoke(
    "Meu objetivo é aprender a construir aplicações de inteligência artificial para desenvolvedores."
)

print(resposta["response"])


## 11. Visualizando o resumo

Podemos consultar o conteúdo armazenado pela memória:

```python
memory.load_memory_variables({})
```

Diferentemente de uma memória de buffer, aqui esperamos encontrar um **resumo do histórico**, e não necessariamente todas as mensagens completas.


In [ ]:
print(memory.load_memory_variables({}))


## 12. Como o resumo é atualizado?

De forma conceitual, podemos imaginar o processo assim:

```text
Interação 1
    ↓
Histórico
    ↓
Resumo

Interação 2
    ↓
Resumo anterior + nova interação
    ↓
Novo resumo

Interação 3
    ↓
Novo resumo + nova interação
    ↓
Resumo atualizado
```

O resumo é continuamente atualizado para incorporar as novas informações relevantes.


## 13. Um ponto importante: Summary Memory usa o LLM

Esse é um dos principais conceitos desta aula.

Na `ConversationBufferMemory`, armazenar o histórico é relativamente simples:

```text
guardar mensagens
```

Na `ConversationSummaryMemory`, existe uma etapa adicional:

```text
histórico
   ↓
LLM
   ↓
resumo
```

Isso significa que a Summary Memory pode consumir chamadas ao modelo para produzir os resumos.

No nosso caso, essas chamadas também são feitas localmente pelo Ollama.


## 14. Vantagens e desvantagens

### Vantagens

- Pode reduzir o tamanho do contexto.
- Pode preservar informações importantes de conversas longas.
- Evita manter todas as mensagens completas no prompt.
- É útil quando uma conversa pode crescer bastante.

### Desvantagens

- O resumo pode perder detalhes.
- A criação do resumo exige processamento do LLM.
- Informações muito específicas podem desaparecer durante a sumarização.
- A qualidade do resultado depende da capacidade do modelo de resumir corretamente.

Por isso, Summary Memory não significa simplesmente "guardar tudo de forma melhor". Existe um **trade-off entre quantidade de contexto e compactação**.


## 15. Comparando as três memórias

| Memória | O que mantém | Característica |
|---|---|---|
| `ConversationBufferMemory` | Histórico completo | Simples e direto |
| `ConversationBufferWindowMemory` | Últimas interações | Limita o histórico |
| `ConversationSummaryMemory` | Resumo do histórico | Compacta o contexto |

Uma forma simples de pensar:

```text
Buffer  → "guarde tudo"
Window  → "guarde o mais recente"
Summary → "resuma o que aconteceu"
```


## 16. Memória não é aprendizado do modelo

Assim como nas aulas anteriores, é importante diferenciar memória de aprendizado.

Quando usamos:

```python
ConversationSummaryMemory
```

o Qwen não está alterando seus parâmetros.

A aplicação está criando um contexto resumido que será utilizado nas próximas chamadas.

```text
Aplicação
    ↓
Resumo da conversa
    ↓
Prompt
    ↓
Qwen
```

Se a memória for apagada, o resumo também deixa de fazer parte do contexto da aplicação.


## 17. Código completo

Abaixo está o código completo deste estudo.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationChain


# 1. Configurando o modelo local do Ollama
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)


# 2. Criando a memória de resumo
#
# A ConversationSummaryMemory utiliza o próprio LLM para resumir
# o histórico da conversa à medida que novas interações acontecem.
memory = ConversationSummaryMemory(
    llm=llm,
    return_messages=True
)


# 3. Criando a conversa usando a memória
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)


# 4. Primeira interação
resposta = conversation.invoke(
    "Meu nome é Felipe e estou estudando inteligência artificial."
)

print("Resposta 1:")
print(resposta["response"])


# 5. Segunda interação
resposta = conversation.invoke(
    "Estou estudando Python, LangChain e desenvolvimento de aplicações com LLMs."
)

print("\nResposta 2:")
print(resposta["response"])


# 6. Terceira interação
resposta = conversation.invoke(
    "Para meus estudos, estou usando Ollama com o modelo Qwen 3 4B localmente."
)

print("\nResposta 3:")
print(resposta["response"])


# 7. Quarta interação
resposta = conversation.invoke(
    "Meu objetivo é aprender a construir aplicações de inteligência artificial para desenvolvedores."
)

print("\nResposta 4:")
print(resposta["response"])


# 8. Visualizando o resumo atual da memória
print("\nResumo atual da memória:")
print(memory.load_memory_variables({}))
